# Personalized reward features: Kaggle training
Settings: **Accelerator = GPU (T4 or P100)**, **Internet = On** (for `git clone`). Add your uploaded `data/` dataset via *Add Input*.

In [ ]:
# 1) Get the code. EDIT the repo URL. For a PRIVATE repo, add a Kaggle Secret named GITHUB_TOKEN
#    (Add-ons > Secrets) and uncomment the 3 lines below.
REPO = "https://github.com/<user>/<repo>.git"
# from kaggle_secrets import UserSecretsClient
# tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
# REPO = f"https://{tok}@github.com/<user>/<repo>.git"
# Leave the old checkout BEFORE deleting it. Re-running this cell from inside proj/ otherwise leaves the
# kernel in a deleted directory ('getcwd: cannot access parent directories').
%cd /kaggle/working
!rm -rf /kaggle/working/proj && git clone --depth 1 $REPO /kaggle/working/proj
%cd /kaggle/working/proj
!ls scripts   # should list kaggle_run.py and kaggle_train.ipynb

In [ ]:
# 2) Sanity: GPU visible? data found?
import torch; print(torch.__version__, "cuda:", torch.cuda.is_available())
!python scripts/kaggle_run.py --stage train --dry-run   # prints the data root + the exact command

In [ ]:
# 3) Unit tests (the 5 torch tests have never been run on real torch; this is their first run)
!python scripts/kaggle_run.py --stage tests

In [ ]:
# 4) Smoke run: ~2000 steps. Check val acc rises toward the ceiling (~0.8) and Spearman toward 1.
!python scripts/kaggle_run.py --stage train --tasks reach-v3 --steps 2000 --tag smoke

In [ ]:
# 5) Full training, both tasks (K=16, transformer)
!python scripts/kaggle_run.py --stage train --tasks reach-v3 pick-place-v3 --steps 20000

In [ ]:
# 6) Few-shot adaptation on unseen users (use the same --K/--temporal/--tag as training)
!python scripts/kaggle_run.py --stage adapt --tasks reach-v3 pick-place-v3

In [ ]:
# 7) Bundle checkpoints + result tables/logs -> /kaggle/working/outputs.zip (download from the Output panel)
!python scripts/kaggle_run.py --stage bundle